# SpecDist — Lightning AI Studio

**Persistent storage · T4-equivalent (free 22-35 h/month) or L40S (paid) · VS Code environment**

| Cell | What it does | Time |
|------|-------------|------|
| 0. Bootstrap | First-time: clone → deps → auth → run | ~3 min + model dl (once) |
| 1. Resume | After GPU swap or closing browser | ~1 min |
| 2. Monitor | State + log tail (auto-refresh option) | instant |

---

### Before you start
1. **Enable GPU**: Studio Settings (⚙, top-right) → Compute → switch to **T4** (free tier) or **L40S** (paid)
2. **Set secrets** (one-time, persist across sessions): Studio Settings → **Environment Variables**:
   - `GITHUB_TOKEN` — required if repo is private
   - `WANDB_API_KEY` — from https://wandb.ai/authorize
   - `HF_TOKEN` — optional (Qwen3 models are public)
3. **Run all cells**: Shift+Enter or the Run All button

### CONFIG options

| CONFIG | Teacher | Steps | VRAM | Time | When to use |
|--------|---------|-------|------|------|-------------|
| `kaggle` | Qwen3-8B (4-bit NF4) | 1000 | ~7.8 GB | ~5-8 h | **Free T4** — best signal per credit |
| `colab` | Qwen3-4B (BF16) | 500 | ~10.7 GB | ~4 h | Free T4 safe fallback |
| `colab_a100` | Qwen3-8B (BF16) | 2000 | ~19 GB | ~2-3 h | **Paid L40S** (48 GB VRAM) |

### Why Lightning AI is different from Colab/Kaggle

- **Storage persists across sessions** — `~/specdist/` is never wiped. Models download once (~10 min for 8B), then load instantly every session.
- **No session time limit** — GPU stays on until you pause it or credits run out.
- **VS Code native** — open `gbv-research/` as a workspace; edit YAMLs, configs, losses in the IDE while the pipeline runs in the terminal.

### Free credit budget (15 credits = ~$15 USD/month)

| GPU | Cost/h | Free hours/month | Full runs/month |
|-----|--------|-----------------|-----------------|
| T4-equivalent | ~$0.50-0.65/h | 22-30 h | 2-3 × `kaggle` config (~6 h each) |
| L40S (48 GB) | ~$1.10-1.50/h | n/a (paid) | — |

**Tip:** Switch back to the free CPU machine when not training — GPU credits only tick while the GPU instance is active.

### First-session model download
The 8B teacher (~16 GB) downloads to `~/specdist/hf_cache/` on the first run (~10 min).
Every subsequent session it loads from cache instantly — no re-download ever.
Storage is counted against your 10-50 GB Studio quota.

Full setup guide: `docs/KAGGLE.md` (same 8B NF4 strategy applies here)

In [ ]:
# =============================================================================
# Cell 0 — BOOTSTRAP  (first session, or after cloning a new Studio)
# Lightning AI: secrets live in Studio Settings → Environment Variables.
# Storage at ~/specdist/ persists — models cache once, checkpoints never lost.
# =============================================================================

import os, subprocess, sys

# -- Edit these ---------------------------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR     = os.path.expanduser("~/Distill-Spec-Research")
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = os.path.expanduser("~/specdist")   # persistent across sessions

# CONFIG options:
#   kaggle     — Qwen3-8B (4-bit NF4, ~7.8 GB VRAM, 1000 steps, ~5-8 h)  ← free T4
#   colab      — Qwen3-4B (BF16,      ~10.7 GB VRAM, 500 steps,  ~4 h)   ← safe fallback
#   colab_a100 — Qwen3-8B (BF16,      ~19 GB VRAM,   2000 steps, ~2-3 h) ← paid L40S
CONFIG      = "kaggle"
SMOKE       = False    # True = 10-step crash check (~5 min); run first on a new Studio
BACKGROUND  = True     # True = background; monitor with Cell 2
LOSSES      = None     # None = all losses  |  "kl,ebe" = subset
EXTRA_ARGS  = []
# -----------------------------------------------------------------------------

def _prereq_token(name):
    # Secrets from Studio Settings -> Environment Variables (no special API needed).
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh:
    os.environ["GITHUB_TOKEN"] = gh
else:
    print("\u26a0  GITHUB_TOKEN not set — clone will fail for a private repo.")
    print("   Studio Settings (⚙) → Environment Variables → add GITHUB_TOKEN")

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout.strip() or "Already up to date")
    if r.returncode != 0: print("[pull error]", r.stderr.strip())

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

# No mount_drive (Lightning storage is already persistent at ~/specdist/).
# No restore_checkpoints (checkpoints persist — nothing to restore from an external source).
# First session: 8B model downloads to ~/specdist/hf_cache/ (~10 min).
# Every subsequent session: loads from persistent cache instantly.
bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR)
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR,
                 smoke=SMOKE, losses=LOSSES,
                 background=BACKGROUND, extra_args=EXTRA_ARGS)

In [ ]:
# =============================================================================
# Cell 1 — RESUME  (after GPU swap, pausing the Studio, or closing the browser)
# Self-contained — works even if Cell 0 never ran this session.
# Storage persists: checkpoints and model cache are already in ~/specdist/.
# =============================================================================

import os, subprocess, sys

# -- Edit these (must match Cell 0) -------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR     = os.path.expanduser("~/Distill-Spec-Research")
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = os.path.expanduser("~/specdist")
CONFIG       = "kaggle"   # must match Cell 0
# -----------------------------------------------------------------------------

def _prereq_token(name):
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh: os.environ["GITHUB_TOKEN"] = gh

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout.strip() or "Already up to date")

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR)
print(f"\nResuming {CONFIG} — completed steps are skipped automatically.\n")
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR, background=True)

In [ ]:
# =============================================================================
# Cell 2 — MONITOR  (safe to run any time, including while pipeline runs)
# AUTO_REFRESH = True → live tail; interrupt cell to stop.
# =============================================================================
import os, sys

REPO_DIR     = os.path.expanduser("~/Distill-Spec-Research")
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = os.path.expanduser("~/specdist")
CONFIG       = "kaggle"   # must match Cell 0/1
AUTO_REFRESH = False      # True = live tail loop (interrupt cell to stop)
REFRESH_SECS = 20

sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import monitor

monitor(STORAGE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)